# Rally E2B Browser Export

Consumes the Rally E2B two-stage SFT Kaggle output and publishes direct/full, direct/text, RP merged, RP/full, and RP/text artifacts to private Hugging Face repos.

In [ ]:
import os, platform, shutil
from pathlib import Path

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
print('input_dirs=', [str(p) for p in Path('/kaggle/input').glob('*')])
try:
    import torch
    print('torch=', torch.__version__)
    print('cuda_available=', torch.cuda.is_available())
    print('gpu_count=', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        major, minor = torch.cuda.get_device_capability(i)
        print(f'gpu_{i}=', props.name, round(props.total_memory / 1024**3, 2), 'GiB', f'sm_{major}{minor}')
except Exception as exc:
    print('torch_probe_error=', repr(exc))


In [ ]:
import os, subprocess, sys, time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
secret_token = ''
for attempt in range(5):
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret('HF_TOKEN')
        break
    except Exception as exc:
        print('hf_secret_attempt_failed=', attempt + 1, type(exc).__name__)
        time.sleep(3)
if secret_token and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret_token
if secret_token and not os.environ.get('HUGGING_FACE_HUB_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = secret_token
print('hf_secret_loaded=', bool(secret_token))

packages = [
    'git+https://github.com/huggingface/transformers.git@c472755e79aac54d675845bff5e5c821c21260af',
    'accelerate>=1.13.0',
    'huggingface_hub[cli]>=1.5.0',
    'hf_transfer>=0.1.9',
    'safetensors>=0.7.0',
    'onnx>=1.19.0',
    'onnx-ir>=0.1.12',
    'onnxruntime>=1.23.0',
    'onnxscript>=0.5.0',
    'onnxconverter-common>=1.14.0',
    'sentencepiece>=0.2.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])


In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = os.environ.get('HERETIC_TO_ONNX_REPO', 'https://github.com/alkahest-ai/heretic-to-onnx.git')
REPO_REF = os.environ.get('HERETIC_TO_ONNX_REF', 'codex/kaggle-heretic-2b-run')
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')

if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])

print('repo=', REPO_DIR)
print('head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
WORK_DIR = Path(os.environ.get('RALLY_EXPORT_WORK_DIR', '/kaggle/working/rally-e2b-browser-export'))
cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts/kaggle_rally_e2b_two_stage_export.py'),
    '--work-dir', str(WORK_DIR),
    '--scratch-dir', os.environ.get('RALLY_EXPORT_SCRATCH_DIR', '/kaggle/temp/rally-e2b-browser-export'),
    '--artifact-name', os.environ.get('RALLY_TWO_STAGE_ARTIFACT_NAME', 'rally-e2b-two-stage-sft'),
    '--direct-full-repo', os.environ.get('RALLY_DIRECT_FULL_REPO', 'thomasjvu/rally-2b'),
    '--direct-text-repo', os.environ.get('RALLY_DIRECT_TEXT_REPO', 'thomasjvu/rally-2b-text'),
    '--rp-merged-repo', os.environ.get('RALLY_RP_MERGED_REPO', 'thomasjvu/rally-2b-rp-a100-b75-merged'),
    '--rp-full-repo', os.environ.get('RALLY_RP_FULL_REPO', 'thomasjvu/rally-2b-rp'),
    '--rp-text-repo', os.environ.get('RALLY_RP_TEXT_REPO', 'thomasjvu/rally-2b-rp-text'),
    '--stage-b-scale', os.environ.get('RALLY_STAGE_B_SCALE', '0.75'),
    '--export-device', os.environ.get('RALLY_EXPORT_DEVICE', 'cpu'),
]
if os.environ.get('RALLY_UPLOAD', '0') != '1':
    cmd.append('--no-upload')
if os.environ.get('RALLY_FULL_EXPORT', '0') != '1':
    cmd.append('--skip-full-packages')
if os.environ.get('RALLY_SKIP_RP', '1') == '1':
    cmd.append('--skip-rp')
if os.environ.get('RALLY_NO_SCORE', '') == '1':
    cmd.append('--no-score')
if os.environ.get('RALLY_DIRECT_TEXT_ONLY', '1') == '1':
    cmd.append('--no-score')
subprocess.check_call(cmd)


In [ ]:
from pathlib import Path
import json, os

report_path = Path(os.environ.get('RALLY_EXPORT_WORK_DIR', '/kaggle/working/rally-e2b-browser-export')) / 'rally-e2b-browser-export-report.json'
print('report_path=', report_path)
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps({
        'ok': report.get('ok'),
        'artifact_dir': report.get('artifact_dir'),
        'selected_merged': report.get('selected_merged'),
        'merged_upload': report.get('merged_upload'),
        'exports': [item.get('target', {}).get('repo_id') for item in report.get('exports', [])],
        'warnings': report.get('warnings', []),
        'scorecard': report.get('scorecard', {}).get('promotion_decision'),
    }, indent=2))
else:
    raise FileNotFoundError(report_path)
